# M5 clustering demo (L1 + L2)

End-to-end demo notebook implementing semantic difficulty calibration (L1) and structural clustering (L2) on the provided `m5_preprocess_data.csv` wide panel.

## 1. Config and imports

This notebook follows the provided instruction spec: compute one feature table, reuse it for L1 difficulty calibration and L2 clustering. The design is pandas-first and CPU-only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import entropy
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

pd.set_option("display.max_columns", 20)
sns.set(style="whitegrid")

RANDOM_STATE = 42
SEASONAL_PERIOD = 7
FORECAST_HORIZON = 28
BACKTEST_STEP = 28
ZERO_RATIO_THRESHOLD = 0.6


## 2. Load data (wide M5 panel)

The CSV ships with the repo and is already in wide format with a daily date column.

In [ ]:
data_path = "m5_preprocess_data.csv"
df_raw = pd.read_csv(data_path, parse_dates=["date"])
df_raw = df_raw.set_index("date").sort_index()
df_raw.head()


## 3. Level-1 preprocess (time integrity + light NA handling)

Reindex to a complete daily grid, lightly fill short gaps for metric robustness, and compute quality flags.

In [ ]:
def preprocess_level1(df_wide: pd.DataFrame, freq: str = "D"):
    """Clean wide panel and compute quality flags.

    Returns
    -------
    df_clean : DataFrame
        Reindexed and lightly filled series.
    qc_df : DataFrame
        Quality metrics per series.
    """
    df = df_wide.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    duplicate_ts_flag = df.index.duplicated().any()
    if duplicate_ts_flag:
        df = df.loc[~df.index.duplicated(keep="first")]

    inferred_freq = pd.infer_freq(df.index)
    if inferred_freq is None:
        inferred_freq = freq
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq=inferred_freq)
    df = df.reindex(full_idx)

    df = df.apply(pd.to_numeric, errors="coerce")
    missing_ratio = df.isna().mean()
    n_obs = df.notna().sum()

    df = df.ffill(limit=7).bfill(limit=1)
    df = df.fillna(0)

    data_quality_issue = (missing_ratio > 0.2) | (n_obs < int(len(full_idx) * 0.5))
    qc_df = pd.DataFrame(
        {
            "n_obs": n_obs,
            "missing_ratio": missing_ratio,
            "duplicate_ts_flag": duplicate_ts_flag,
            "data_quality_issue": data_quality_issue,
        }
    )

    return df, qc_df


df_clean, qc_df = preprocess_level1(df_raw)
df_clean.head()


## 4. Level-2 preprocess (outlier clipping for metrics only)

Use robust MAD-based clipping so feature computation is resilient without altering the main cleaned panel.

In [ ]:
def preprocess_level2_for_metrics(df_clean: pd.DataFrame, k: float = 8.0):
    """Return df_metrics for feature computation with MAD clipping."""
    df_metrics = df_clean.copy()
    eps = 1e-8
    for col in df_metrics.columns:
        series = df_metrics[col]
        median = series.median()
        mad = stats.median_abs_deviation(series, nan_policy="omit")
        mad = mad if np.isfinite(mad) and mad > 0 else series.abs().median() + eps
        lower = median - k * mad
        upper = median + k * mad
        df_metrics[col] = series.clip(lower=lower, upper=upper)
    return df_metrics


df_metrics = preprocess_level2_for_metrics(df_clean)
df_metrics.head()


## 5. Feature computation (single pass)

Build `features_df` once and reuse it for both L1 difficulty calibration and L2 clustering. Includes STL-based seasonality/trend strength, spectral entropy, stability/lumpiness, spikiness, and intermittent demand metrics (ADI/CV²).

In [ ]:
def _stl_components(series: pd.Series, seasonal_period: int):
    stl = STL(series, period=seasonal_period, robust=True)
    res = stl.fit()
    return res.trend, res.seasonal, res.resid


def _strength(component: np.ndarray, resid: np.ndarray):
    var_resid = np.nanvar(resid)
    var_total = np.nanvar(component + resid)
    if var_total <= 0:
        return 0.0
    return max(0.0, 1 - var_resid / var_total)


def _spectral_entropy(series: pd.Series):
    x = series.values
    if len(x) == 0:
        return np.nan
    x = x - np.nanmean(x)
    spectrum = np.fft.rfft(x)
    psd = np.abs(spectrum) ** 2
    psd_sum = psd.sum()
    if psd_sum == 0:
        return np.nan
    psd /= psd_sum
    se = entropy(psd, base=2)
    max_entropy = np.log2(len(psd)) if len(psd) > 0 else 1
    return se / max_entropy


def _stability_lumpiness(series: pd.Series, width: int):
    windows = [series[i : i + width] for i in range(0, len(series), width)]
    window_means = [w.mean() for w in windows if len(w) == width]
    window_vars = [w.var() for w in windows if len(w) == width]
    stability = np.var(window_means) if window_means else np.nan
    lumpiness = np.var(window_vars) if window_vars else np.nan
    return stability, lumpiness


def _spikiness(remainder: np.ndarray):
    eps = 1e-8
    mad = stats.median_abs_deviation(remainder, nan_policy="omit")
    mad = mad if mad > 0 else np.nanmedian(np.abs(remainder)) + eps
    return np.nanpercentile(np.abs(remainder), 99) / (mad + eps)


def _adi_cv2(series: pd.Series):
    events = series > 0
    zero_ratio = 1 - events.mean()
    event_idx = np.flatnonzero(events.values)
    if len(event_idx) > 1:
        adi = np.diff(event_idx).mean()
    else:
        adi = np.inf
    positive = series[series > 0]
    if len(positive) > 1 and positive.mean() > 0:
        cv2 = (positive.std() / positive.mean()) ** 2
    else:
        cv2 = np.inf
    return zero_ratio, adi, cv2


def compute_features(df_metrics: pd.DataFrame, seasonal_period: int = SEASONAL_PERIOD):
    records = []
    for col in df_metrics.columns:
        series = df_metrics[col].astype(float)
        if series.isna().all():
            continue
        try:
            trend, seasonal, resid = _stl_components(series, seasonal_period)
        except Exception:
            trend = pd.Series(np.zeros_like(series), index=series.index)
            seasonal = pd.Series(np.zeros_like(series), index=series.index)
            resid = series - series.mean()

        trend_strength = _strength(trend.values, resid.values)
        seasonality_strength = _strength(seasonal.values, resid.values)
        spectral_entropy = _spectral_entropy(series)
        stability, lumpiness = _stability_lumpiness(series, width=seasonal_period)
        spikiness = _spikiness(resid.values)
        zero_ratio, adi, cv2 = _adi_cv2(series)

        records.append(
            {
                "series_id": col,
                "n_obs": series.notna().sum(),
                "missing_ratio": series.isna().mean(),
                "constant_ratio": (series.diff().abs().fillna(0) == 0).mean(),
                "trend_strength": trend_strength,
                "seasonality_strength": seasonality_strength,
                "spectral_entropy": spectral_entropy,
                "stability": stability,
                "lumpiness": lumpiness,
                "spikiness": spikiness,
                "zero_ratio": zero_ratio,
                "adi": adi,
                "cv2": cv2,
            }
        )
    features_df = pd.DataFrame.from_records(records).set_index("series_id")
    return features_df


features_df = compute_features(df_metrics)
features_df.head()


## 6. L1 calibration via rolling-origin backtest

Use Naive and Seasonal Naive baselines with rolling-origin splits to derive a normalized difficulty score and percentile.

In [ ]:
def _smape(y_true, y_pred):
    eps = 1e-8
    return np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps))


def rolling_origin_backtest_difficulty(
    df_clean: pd.DataFrame, H: int = FORECAST_HORIZON, step: int = BACKTEST_STEP, seasonal_period: int = SEASONAL_PERIOD
):
    results = []
    n_total = len(df_clean)
    for col in df_clean.columns:
        series = df_clean[col].astype(float)
        values = series.values
        folds_naive = []
        folds_snaive = []
        start = int(n_total * 0.6)
        if n_total < H + seasonal_period + 5:
            results.append({"series_id": col, "bt_error_naive": np.nan, "bt_error_snaive": np.nan, "difficulty_score": np.nan})
            continue
        for split in range(start, n_total - H + 1, step):
            train = values[:split]
            test = values[split : split + H]
            if len(train) == 0 or np.all(np.isnan(train)):
                continue
            last_val = pd.Series(train).ffill().iloc[-1]
            naive_fcst = np.repeat(last_val, H)
            if split - seasonal_period >= 0:
                seasonal_history = pd.Series(train).ffill().iloc[-seasonal_period:]
                snaive_fcst = np.tile(seasonal_history.values, int(np.ceil(H / seasonal_period)))[:H]
            else:
                snaive_fcst = naive_fcst
            folds_naive.append(_smape(test, naive_fcst))
            folds_snaive.append(_smape(test, snaive_fcst))

        bt_error_naive = np.median(folds_naive) if folds_naive else np.nan
        bt_error_snaive = np.median(folds_snaive) if folds_snaive else np.nan
        difficulty_score = np.nanmin([bt_error_naive, bt_error_snaive])
        results.append(
            {
                "series_id": col,
                "bt_error_naive": bt_error_naive,
                "bt_error_snaive": bt_error_snaive,
                "difficulty_score": difficulty_score,
            }
        )
    difficulty_df = pd.DataFrame(results).set_index("series_id")
    difficulty_df["difficulty_quantile"] = difficulty_df["difficulty_score"].rank(pct=True, method="average")
    return difficulty_df


difficulty_df = rolling_origin_backtest_difficulty(df_clean)
difficulty_df.head()


## 7. L1 labeling (A/B/C) with data-driven thresholds

Combine backtest difficulty, intermittent demand signals, and data-quality flags. Intermittent thresholds come from the Syntetos/Boylan ADI/CV² scheme (C). Quantile-based difficulty separates A vs B without manual cutoffs.

In [ ]:
def label_L1(features_df: pd.DataFrame, difficulty_df: pd.DataFrame, qc_df: pd.DataFrame):
    df = features_df.join(difficulty_df, how="left").join(qc_df, how="left")

    stability_q = df["stability"].rank(pct=True, method="average")
    lumpiness_q = df["lumpiness"].rank(pct=True, method="average")

    is_C = (df["adi"] > 1.32) | ((df["cv2"] > 0.49) & (df["zero_ratio"] > ZERO_RATIO_THRESHOLD))
    is_B_data = df["data_quality_issue"] | df["duplicate_ts_flag"]
    is_B_inherent = (~is_C) & (df["difficulty_quantile"] >= 0.70)
    is_B_instability = (~is_C) & ((stability_q > 0.80) | (lumpiness_q > 0.80))

    labels = []
    reasons = []
    for idx, row in df.iterrows():
        drivers = []
        if is_C.loc[idx]:
            label = "C"
            drivers.append("high ADI/CV2/zero ratio")
        elif is_B_data.loc[idx] or is_B_inherent.loc[idx] or is_B_instability.loc[idx]:
            label = "B"
            if is_B_data.loc[idx]:
                drivers.append("data quality guardrail")
            if is_B_inherent.loc[idx]:
                drivers.append("high difficulty score")
            if is_B_instability.loc[idx]:
                drivers.append("instability (stability/lumpiness)")
        else:
            label = "A"
            drivers.append("structured (trend/seasonality)")
            drivers.append("low entropy / low lumpiness")
        labels.append(label)
        reasons.append("; ".join(drivers[:3]))

    l1_df = df[["difficulty_score", "difficulty_quantile", "adi", "cv2", "zero_ratio", "stability", "lumpiness"]].copy()
    l1_df["L1_label"] = labels
    l1_df["L1_reason_top3"] = reasons
    return l1_df


l1_df = label_L1(features_df, difficulty_df, qc_df)
l1_df["L1_label"].value_counts()


## 8. L2 structural clustering with auto-k

Pool A and C separately, standardize a structural feature vector, run PCA, and pick k via a composite silhouette/Davies–Bouldin/size score. Use MiniBatchKMeans for scalability.

In [ ]:
def _score_k(pca_features: np.ndarray, labels: np.ndarray, min_cluster_size: int = 20):
    unique, counts = np.unique(labels, return_counts=True)
    size_penalty = (counts < min_cluster_size).sum() / len(unique)
    sil = silhouette_score(pca_features, labels) if len(unique) > 1 else -np.inf
    db = davies_bouldin_score(pca_features, labels) if len(unique) > 1 else np.inf
    return sil - 0.2 * db - 0.5 * size_penalty, sil, db, size_penalty


def cluster_L2(features_df: pd.DataFrame, l1_df: pd.DataFrame, pool: str):
    assert pool in {"A", "C"}
    pool_mask = l1_df["L1_label"] == pool
    pool_features = features_df.loc[pool_mask]
    if pool_features.empty:
        return pd.DataFrame(columns=["series_id", "L2_pool", "cluster_id", "cluster_size", "cluster_cohesion"])

    structural_cols = ["seasonality_strength", "trend_strength", "spectral_entropy"]
    X = pool_features[structural_cols].fillna(pool_features[structural_cols].median())
    X_scaled = StandardScaler().fit_transform(X)
    n_components = min(5, X_scaled.shape[1], X_scaled.shape[0])
    X_pca = PCA(n_components=n_components, random_state=RANDOM_STATE).fit_transform(X_scaled)

    n_pool = len(pool_features)
    k_min = max(2, min(5, n_pool))
    k_max = min(30, int(np.sqrt(n_pool)) if n_pool > 1 else 1)
    if k_max < k_min:
        k_max = k_min

    best = None
    for k in range(k_min, k_max + 1):
        model = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE)
        labels = model.fit_predict(X_pca)
        score, sil, db, penalty = _score_k(X_pca, labels)
        if best is None or score > best[0]:
            best = (score, k, model, labels, sil, db, penalty)

    _, best_k, best_model, best_labels, best_sil, best_db, best_penalty = best
    distances = best_model.transform(X_pca)
    dists_to_centroid = distances[np.arange(len(best_labels)), best_labels]

    l2_df = pool_features.copy()
    l2_df["L2_pool"] = f"{pool}_pool"
    l2_df["cluster_id"] = best_labels
    l2_df["cluster_size"] = l2_df.groupby("cluster_id")["cluster_id"].transform("size")
    l2_df["cluster_cohesion"] = dists_to_centroid
    l2_df["silhouette_score"] = best_sil
    l2_df["davies_bouldin_score"] = best_db
    l2_df["tiny_cluster_penalty"] = best_penalty
    return l2_df


l2A_df = cluster_L2(features_df, l1_df, pool="A")
l2C_df = cluster_L2(features_df, l1_df, pool="C")

l2A_df.head()


## 9. Routing decisions

Apply deterministic routing rules to decide global/local scope and deterministic/probabilistic forecast type.

In [ ]:
def route_models(l1_df: pd.DataFrame, l2A_df: pd.DataFrame, l2C_df: pd.DataFrame, min_global_size: int = 50):
    routing_records = []
    cohesion_cutoff_A = l2A_df["cluster_cohesion"].quantile(0.7) if not l2A_df.empty else np.inf
    cohesion_cutoff_C = l2C_df["cluster_cohesion"].quantile(0.7) if not l2C_df.empty else np.inf

    l2_lookup = pd.concat([l2A_df[["cluster_id", "cluster_size", "cluster_cohesion", "L2_pool"]], l2C_df[["cluster_id", "cluster_size", "cluster_cohesion", "L2_pool"]]])

    for series_id, row in l1_df.iterrows():
        label = row["L1_label"]
        model_scope = "local"
        forecast_type = "deterministic"
        global_group = None
        explain = []

        if label == "B":
            explain.append("high difficulty or data issue")
        else:
            if label == "C":
                forecast_type = "probabilistic"
                pool_df = l2C_df
                cohesion_cutoff = cohesion_cutoff_C
            else:
                pool_df = l2A_df
                cohesion_cutoff = cohesion_cutoff_A

            if series_id in pool_df.index:
                cluster_row = pool_df.loc[series_id]
                if (cluster_row["cluster_size"] >= min_global_size) and (
                    cluster_row["cluster_cohesion"] <= cohesion_cutoff
                ):
                    model_scope = "global"
                    global_group = int(cluster_row["cluster_id"])
                    explain.append("cluster strong enough for global model")
                else:
                    explain.append("cluster too small/loose -> local")

        routing_records.append(
            {
                "series_id": series_id,
                "quality_bucket": label,
                "model_scope": model_scope,
                "global_group_id": global_group,
                "forecast_type": forecast_type,
                "explain": "; ".join(explain) if explain else "default path",
            }
        )

    routing_df = pd.DataFrame(routing_records).set_index("series_id")
    return routing_df


routing_df = route_models(l1_df, l2A_df, l2C_df, min_global_size=20)
routing_df.head()


## 10. Dashboards and diagnostics

Visual summaries for difficulty distribution, structural features, and clustering layouts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Difficulty histogram
sns.histplot(l1_df["difficulty_score"].dropna(), kde=True, ax=axes[0], color="steelblue")
axes[0].axvline(l1_df["difficulty_score"].quantile(0.70), color="orange", linestyle="--", label="B boundary (70% q)")
axes[0].set_title("Difficulty score distribution")
axes[0].legend()

# Seasonality vs trend
sns.scatterplot(
    data=l1_df.join(features_df), x="seasonality_strength", y="trend_strength", hue="L1_label", palette="Set2", ax=axes[1]
)
axes[1].set_title("Seasonality vs Trend strength")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ["spectral_entropy", "stability", "lumpiness"]):
    sns.boxplot(data=l1_df.join(features_df), x="L1_label", y=col, ax=ax, palette="Set2")
    ax.set_title(f"{col} by L1 label")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (df_pool, title) in zip(axes, [(l2A_df, "A pool"), (l2C_df, "C pool")]):
    if df_pool.empty:
        ax.text(0.5, 0.5, f"No {title} clusters", ha="center")
        continue
    structural_cols = ["seasonality_strength", "trend_strength", "spectral_entropy"]
    X = df_pool[structural_cols].fillna(df_pool[structural_cols].median())
    X_scaled = StandardScaler().fit_transform(X)
    X_pca = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df_pool["cluster_id"], cmap="tab20", s=30)
    ax.set_title(f"PCA view of {title} clusters")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    legend1 = ax.legend(*scatter.legend_elements(), title="cluster_id", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 11. Sample routing table

Peek at a few series with their label, drivers, and routing decision.

In [ ]:
summary_df = routing_df.join(l1_df[["L1_label", "L1_reason_top3"]])
summary_df.head(10)


## 12. Notes and alternatives

- STL feature formulas follow FPP3 (trend/seasonality strength via variance ratios).
- Rolling-origin evaluation mirrors the cross-validation protocol described in Forecasting: Principles and Practice.
- Intermittent demand ADI/CV² thresholds come from the Syntetos/Boylan scheme used by sktime.
- Clustering k selection uses silhouette and Davies–Bouldin metrics from scikit-learn with a size penalty; extend with gap statistic if needed.